In [1]:
import pandas as pd

titanic = pd.read_csv("../data/titanic_propre.csv")
titanic.shape

(889, 10)

In [2]:
survie_par_classe_et_sexe = titanic.pivot_table(
    values="survived", index="pclass", columns="sex", aggfunc="mean"
).round(3)
survie_par_classe_et_sexe

sex,female,male
pclass,,
1,0.967,0.369
2,0.921,0.157
3,0.500,0.135


**Réponse :** Oui, dans les trois classes. En 1re classe, 96,7 % des femmes ont survécu contre 36,9 % des hommes. En 2e classe, 92,1 % contre 15,7 %. En 3e classe, 50,0 % contre 13,5 %. L'écart est le plus faible en 3e classe, mais il reste très important.

In [3]:
titanic["tranche_age"] = pd.cut(
    titanic["age"],
    bins=[0, 12, 18, 60, 100],
    labels=["enfant", "ado", "adulte", "senior"],
)

survie_par_tranche_age = titanic.groupby("tranche_age", observed=True).agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(3)
survie_par_tranche_age

,passagers,taux_survie
tranche_age,,
enfant,69,0.580
ado,70,0.429
adulte,729,0.365
senior,21,0.190


**Réponse :** Oui, les enfants ont été favorisés : 58,0 % des enfants (12 ans et moins) ont survécu, contre 42,9 % des adolescents, 36,5 % des adultes et 19,0 % des seniors. Le taux de survie baisse avec l'âge. Attention : les seniors ne sont que 21, et 177 âges ont été remplacés par une médiane à l'exercice 08.

In [4]:
prix_moyen_par_port = (
    titanic.groupby("embark_town")["fare"].mean().round(2).sort_values(ascending=False)
)
print(prix_moyen_par_port)
print()

repartition_classes_par_port = pd.crosstab(
    titanic["embark_town"], titanic["pclass"], normalize="index"
).round(3)
print(repartition_classes_par_port)
print()

prix_moyen_par_port_et_classe = (
    titanic.groupby(["embark_town", "pclass"])["fare"].mean().round(2).unstack()
)
print(prix_moyen_par_port_et_classe)

embark_town
Cherbourg      59.95
Southampton    27.08
Queenstown     13.28
Name: fare, dtype: float64

pclass           1      2      3
embark_town                     
Cherbourg    0.506  0.101  0.393
Queenstown   0.026  0.039  0.935
Southampton  0.197  0.255  0.548

pclass            1      2      3
embark_town                      
Cherbourg    104.72  25.36  11.21
Queenstown    90.00  12.35  11.18
Southampton   70.36  20.33  14.64


**Réponse :** Cherbourg a le prix moyen le plus élevé (59,95), devant Southampton (27,08) et Queenstown (13,28). La répartition des classes l'explique en grande partie : 50,6 % des passagers de Cherbourg sont en 1re classe, contre 19,7 % à Southampton et 2,6 % à Queenstown. Mais pas totalement : à classe égale, les billets de 1re classe coûtent aussi plus cher à Cherbourg (104,72) qu'à Southampton (70,36).

In [5]:
survie_selon_voyage_seul = titanic.groupby("alone").agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(3)
print(survie_selon_voyage_seul)
print()

survie_selon_sexe_et_voyage_seul = (
    titanic.groupby(["sex", "alone"])["survived"].mean().round(3).unstack()
)
print(survie_selon_sexe_et_voyage_seul)

       passagers  taux_survie
alone                        
False        354        0.506
True         535        0.301

alone   False  True 
sex                 
female  0.713  0.782
male    0.271  0.156


**Réponse :** Globalement, voyager seul diminuait les chances : 30,1 % de survie contre 50,6 % pour les passagers accompagnés. Mais l'effet dépend du sexe : chez les hommes, 15,6 % de survie seuls contre 27,1 % accompagnés ; chez les femmes, 78,2 % seules contre 71,3 % accompagnées. L'écart global mélange donc plusieurs effets, il faut le lire avec prudence.